## **Data clean & Features extraction**

### Environment Setup

This section initializes the essential libraries for the NLP pipeline:

* **Gensim**: Used for **Collocation Detection** (Bigrams) to recognize compound words like "credit_card".
* **NLTK**: Handles text preprocessing (Lemmatization, Stopwords) and **VADER** sentiment analysis.
* **Scikit-Learn**: Performs **Stratified Train/Test splitting** to preserve class distribution.
* **System Tools (`gc`, `os`)**: Manages memory deallocation and file paths, crucial for processing large datasets in Colab.

In [ ]:
!pip install gensim

import pandas as pd
import numpy as np
import os
import re
import time
import gc
import nltk
import gensim
from gensim.models import Phrases
from gensim.models.phrases import Phraser
from nltk.sentiment import SentimentIntensityAnalyzer
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from tqdm.auto import tqdm
from google.colab import drive
from sklearn.model_selection import train_test_split

### NLP Preprocessing & Feature Engineering

Following dataset isolation, raw unstructured text must be transformed into structured data interpretable by Machine Learning algorithms. This phase is articulated into four fundamental steps:

#### 1. Text Normalization (Cleaning)
To reduce dimensionality and mitigate "noise" within the data, we applied a standard cleaning pipeline:
* **Lowercasing & Regex Cleaning:** Conversion of all text to lowercase and removal of digits and non-alphabetic characters via Regular Expressions.
* **Stopwords Removal:** Elimination of high-frequency common words (articles, prepositions) devoid of significant semantic value, utilizing the NLTK corpus.
* **Lemmatization:** Unlike *stemming* (which merely truncates words), we employed the `WordNetLemmatizer`. This reduces words to their linguistic root (e.g., "running" $\to$ "run", "better" $\to$ "good"), thereby preserving semantic meaning.

#### 2. Bigram Recognition (Contextual Awareness)
Single-word analysis (unigrams) often fails to capture context. Utilizing `Gensim Phrases`, we statistically identified words that frequently co-occur, merging them into single tokens (e.g., "credit" + "card" becomes `credit_card`). This significantly enhances the quality of the subsequent Topic Modeling.

#### 3. Stylometric Feature Extraction (Meta-data)
Prior to text normalization, we extracted structural information regarding the "form" of the review, which often serves as a proxy for emotional intensity:
* **Length:** Word and character counts (longer reviews may indicate more complex experiences).
* **Punctuation:** The frequency of exclamation marks (`!`) and question marks (`?`) is calculated to detect signals of anger, joy, or confusion.
* **Capitalization Ratio:** The percentage of uppercase characters is computed as an indicator of "tone of voice" (e.g., online shouting).

#### 4. Lexical Sentiment Analysis (VADER)
We integrated a baseline sentiment score using **VADER** (Valence Aware Dictionary and sEntiment Reasoner). Unlike trainable ML models, VADER is a rule-based approach optimized for social media contexts; it is capable of handling negation and intensity, providing a normalized `compound score` between -1 (negative) and +1 (positive) for each review.

In [ ]:
# ==============================================================================
# 1. SETUP AND DATA LOADING
# ==============================================================================
print("[INFO] Phase 1: Setup and Loading...")

# Mount Drive
if not os.path.exists('/content/drive'):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except ImportError:
        pass

# Download necessary NLTK resources
# 'punkt': tokenizer; 'wordnet': lemmatizer; 'vader_lexicon': sentiment analysis
nltk_resources = ['stopwords', 'wordnet', 'omw-1.4', 'vader_lexicon', 'punkt']
for res in nltk_resources:
    nltk.download(res, quiet=True)

# Path Configuration
DRIVE_PATH = '/content/drive/MyDrive/MAGISTRALE/Text_Mining/Datasets'
FILE_NAME = 'df_joined.csv'
FULL_PATH = os.path.join(DRIVE_PATH, FILE_NAME)

# Enable tqdm for pandas
tqdm.pandas()

try:
    print("[INFO] Loading Dataset...")
    df_final = pd.read_csv(FULL_PATH)

    # Standardize text column name
    if 'text' in df_final.columns and 'text_review' not in df_final.columns:
         df_final = df_final.rename(columns={'text': 'text_review'})

    # Initial removal of null values
    df_final = df_final.dropna(subset=['text_review'])
    print(f"[INFO] Dataset loaded successfully: {len(df_final)} rows.")
except FileNotFoundError:
    print(f"[ERROR] Critical Error: File {FULL_PATH} not found. Check the path.")
    exit()

# ==============================================================================
# 2. TEXT CLEANING (For TF-IDF / Topic Modeling)
# ==============================================================================
print("\n[INFO] Phase 2: Basic Text Cleaning...")

STOP_WORDS = set(stopwords.words('english'))
LEMMATIZER = WordNetLemmatizer()

def clean_text_basic(text):
    """
    Cleans text by keeping only alphabetic characters, converting to lowercase,
    removing stopwords, and applying lemmatization.
    """
    if pd.isna(text) or not isinstance(text, str):
        return ""

    # Regex: keep only alphabetic characters (a-z), convert to lower
    text_cleaned = re.sub(r'[^a-z\s]', '', text.lower())
    tokens = text_cleaned.split()

    # Lemmatization and Stopwords filtering (removing words <= 2 chars as noise)
    cleaned_tokens = [
        LEMMATIZER.lemmatize(w)
        for w in tokens
        if w not in STOP_WORDS and len(w) > 2
    ]
    return " ".join(cleaned_tokens)

print("   -> Applying text cleaning...")
df_final['text_clean'] = df_final['text_review'].progress_apply(clean_text_basic)

# Remove rows that became empty after cleaning
initial_len = len(df_final)
df_final = df_final[df_final['text_clean'].str.strip().astype(bool)]
print(f"[INFO] Rows removed (empty post-cleaning): {initial_len - len(df_final)}")

# ==============================================================================
# 3. TOPIC MODELING PREPARATION (Bigrams)
# ==============================================================================
print("\n[INFO] Phase 3: Bigram Generation...")

# Fast tokenization
tokens_list = df_final['text_clean'].str.split().tolist()

print("   -> Training Bigram model (Gensim)...")
# Gensim detects common compound words (e.g., "customer_service", "credit_card")
phrases = Phrases(tokens_list, min_count=5, threshold=10)
bigram_model = Phraser(phrases)

# Apply bigrams to the dataframe
df_final['topic_text_clean'] = [" ".join(bigram_model[doc]) for doc in tokens_list]

# Memory cleanup
del tokens_list
gc.collect()

# Reset index to ensure alignment for subsequent steps
df_final = df_final[df_final['topic_text_clean'].str.strip().astype(bool)]
df_final = df_final.reset_index(drop=True)
print(f"[INFO] Index reset. Final valid rows: {len(df_final)}")

# ==============================================================================
# 4. STRUCTURAL FEATURE EXTRACTION
# ==============================================================================
print("\n[INFO] Phase 4: Structural Features Extraction...")

# Word and Character counts
df_final['review_len_words'] = df_final['text_review'].astype(str).apply(lambda x: len(x.split()))
df_final['review_len_chars'] = df_final['text_review'].astype(str).apply(len)

# Punctuation counts (proxy for emotion/urgency)
df_final['num_exclamations'] = df_final['text_review'].astype(str).apply(lambda x: x.count('!'))
df_final['num_questions'] = df_final['text_review'].astype(str).apply(lambda x: x.count('?'))

# Capitalization Ratio (proxy for anger or shouting)
def get_caps_ratio(text):
    s = str(text)
    if not s or len(s) == 0:
        return 0.0
    return sum(1 for c in s if c.isupper()) / len(s)

df_final['caps_ratio'] = df_final['text_review'].progress_apply(get_caps_ratio)

# ==============================================================================
# 5. VADER SENTIMENT ANALYSIS
# ==============================================================================
print("\n[INFO] Phase 5: VADER Sentiment Analysis...")

sia = SentimentIntensityAnalyzer()

print("   -> Calculating Polarity Scores...")
# Calculate scores on the RAW text to preserve punctuation/caps nuances
vader_res = df_final['text_review'].progress_apply(lambda x: sia.polarity_scores(str(x)))

# Convert list of dictionaries to DataFrame
vader_df = pd.DataFrame(vader_res.tolist())

# Direct assignment (safe due to previous reset_index)
df_final['vader_neg'] = vader_df['neg']
df_final['vader_neu'] = vader_df['neu']
df_final['vader_pos'] = vader_df['pos']
df_final['vader_compound'] = vader_df['compound']

# ==============================================================================
# 6. FINAL EXPORT
# ==============================================================================
OUTPUT_FILE = os.path.join(DRIVE_PATH, 'df_final.csv')
print(f"\n[INFO] Saving processed file to: {OUTPUT_FILE}")

try:
    df_final.to_csv(OUTPUT_FILE, index=False)
    print(f"[INFO] File saved successfully. Dimensions: {df_final.shape}")
except Exception as e:
    print(f"[ERROR] Error during save: {e}")

print("\n[INFO] PIPELINE COMPLETED")

# ==============================================================================
# 7. OUTPUT SUMMARY
# ==============================================================================
print("\n[INFO] Phase 7: Output Summary")

# 1. Print all column names
print(f"Total Columns Generated: {len(df_final.columns)}")
print("Column List:")
print(df_final.columns.tolist())

# 2. Preview key columns
print("\nPreview of first 3 rows (Key columns):")
cols_to_show = ['text_review', 'text_clean', 'topic_text_clean']
print(df_final[cols_to_show].head(3))

# 3. Rapid statistics on new numerical features
print("\nDescriptive Statistics:")
stats_cols = ['review_len_words', 'review_len_chars', 'num_exclamations', 'num_questions', 'caps_ratio', 'vader_compound']
print(df_final[stats_cols].describe())

### Data Dictionary: Generated Features

The final processed dataset (`df_final.csv`) comprises the following attributes, categorized by their analytical role:

#### 1. Textual Data
* `text_review`: The original, raw text of the review.
* `text_clean`: Normalized text pre-processed for NLP tasks (lowercasing, punctuation removal, lemmatization, stop-words removal).
* `topic_text_clean`: Text optimized specifically for Topic Modeling. This version includes **bigrams** (e.g., _"customer_service"_ joined by underscores) to preserve semantic context.

#### 2. Structural Features (Meta-data)
These features capture the structural composition and expressive intensity of the review:
* `review_len_words`: Total word count of the review.
* `review_len_chars`: Total character count of the review.
* `num_exclamations`: Count of exclamation marks (`!`). This is often correlated with high-arousal emotions (e.g., anger or excitement).
* `num_questions`: Count of question marks (`?`). Indicative of confusion, uncertainty, or rhetorical questioning.
* `caps_ratio`: The ratio of uppercase characters to the total text length. Serves as a proxy for "tone of voice" (e.g., online shouting).

#### 3. Sentiment Analysis (VADER)
Scores computed using the *Valence Aware Dictionary and sEntiment Reasoner* (VADER) algorithm:
* `vader_neg`: Negativity score (Range: 0.0 - 1.0).
* `vader_neu`: Neutrality score (Range: 0.0 - 1.0).
* `vader_pos`: Positivity score (Range: 0.0 - 1.0).
* `vader_compound`: Normalized composite score. Ranges from **-1** (Extremely Negative) to **+1** (Extremely Positive).

### Dataset Partitioning (Train/Test Split)

This phase prepares the final datasets for modeling:

1.  **Target Validation**: The `stars_review` column is cleaned of any null values and cast to integers to ensure data integrity.
2.  **Stratified Split**: The data is divided into **Train (80%)** and **Test (20%)** sets. We employ **stratification** to ensure the class distribution (1-5 stars) remains identical in both subsets, avoiding sampling bias.
3.  **Export**: The resulting partitions are saved as static CSV files for reproducible modeling.

In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

# Note: Uses 'df_final' from the previous chunk.

print("\n" + "="*50)
print("[INFO] CHUNK 2: TRAIN/TEST SPLIT INITIATED")
print("="*50)

# Configuration
DRIVE_PATH = '/content/drive/MyDrive/MAGISTRALE/Text_Mining/Datasets'
TARGET_COL = 'stars_review'

# ------------------------------------------------------------------------------
# 1. DATA INTEGRITY CHECK (Target Column)
# ------------------------------------------------------------------------------
print(f"[INFO] Verifying null values in target column '{TARGET_COL}'...")

# Count NaNs
nan_count = df_final[TARGET_COL].isna().sum()

if nan_count > 0:
    print(f"[WARNING] Found {nan_count} missing values (NaN) in target variable.")
    print("   -> Removing rows with null target...")

    # Drop rows where target is NaN
    initial_len = len(df_final)
    df_final = df_final.dropna(subset=[TARGET_COL])
    dropped_len = initial_len - len(df_final)

    print(f"[INFO] Rows removed: {dropped_len}. New dataset size: {len(df_final)}")
else:
    print("[INFO] No null values found. Target variable is clean.")

# Ensure target is integer type (Classification requirement)
df_final[TARGET_COL] = df_final[TARGET_COL].astype(int)

# ------------------------------------------------------------------------------
# 2. STRATIFIED SPLIT EXECUTION
# ------------------------------------------------------------------------------
print(f"\n[INFO] Partitioning dataset (Target: {TARGET_COL})...")

# 80% Train, 20% Test - Stratified by star rating
# Stratification ensures that the class distribution (1-5 stars) is preserved in both sets.
train_df, test_df = train_test_split(
    df_final,
    test_size=0.20,
    random_state=42,
    stratify=df_final[TARGET_COL]
)

print(f"[INFO] Train Set Dimensions: {train_df.shape}")
print(f"[INFO] Test Set Dimensions:  {test_df.shape}")

# ------------------------------------------------------------------------------
# 3. FINAL FILE EXPORT
# ------------------------------------------------------------------------------
TRAIN_FILE = 'train_dataset.csv'
TEST_FILE = 'test_dataset.csv'

print(f"\n[INFO] Saving datasets to {DRIVE_PATH}...")

train_path = os.path.join(DRIVE_PATH, TRAIN_FILE)
test_path = os.path.join(DRIVE_PATH, TEST_FILE)

train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

print("\n" + "="*50)
print("[SUCCESS] PRE-PROCESSING PIPELINE COMPLETED")
print("="*50)
print(f"Files ready for modeling:")
print(f"   1. {TRAIN_FILE}")
print(f"   2. {TEST_FILE}")